# Coding-docs RAG poisoning - Week 2 MAIN notebook (clean baseline)

This is the **main** notebook (Drive, GPU, API key). Unit tests run in the separate
**test** notebook (`week2_tests_colab.ipynb`: CPU, no Drive, no keys).

| Step | Runtime | Cells |
|---|---|---|
| Day 4 | **L4 GPU** | setup, freeze retrieval, build prompts, pilot |
| Days 5-6 | **L4 GPU** | full runs, one model at a time; GPT-5.4 mini batch (any runtime) |
| Day 6 | any | export answers -> test notebook |
| Day 7 | any | score, hand-check sample |

Run the setup cell first in every session. Every run resumes where it stopped.

In [ ]:
# ===== SETUP: run first in every session =====
REPO_URL = "https://github.com/MrunalKotkar/Code-Rag-Defense"
REPO_DIR = "/content/repo"
NEED_OLLAMA = True          # True when this session runs local models
OLLAMA_MODELS = ['llama3.1:8b', 'qwen2.5-coder:7b', 'llama3.2:3b']

import os, subprocess, time
from google.colab import drive

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

if "<" in REPO_URL:
    raise SystemExit("Set REPO_URL (first line of this cell) to your GitHub repo's HTTPS URL, then re-run.")

drive.mount("/content/drive")
if os.path.isdir(REPO_DIR):
    sh(f"git -C {REPO_DIR} pull --ff-only")
else:
    sh(f"git clone {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
sh("pip install -q -r coding_track/requirements-colab.txt")

if NEED_OLLAMA:
    sh("apt-get -qq install -y pciutils zstd > /dev/null")   # pciutils helps Ollama find a GPU
    sh("curl -fsSL https://ollama.com/install.sh | sh")
    env = dict(os.environ, OLLAMA_CONTEXT_LENGTH="16384")
    subprocess.Popen(["ollama", "serve"], env=env,
                     stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
    time.sleep(5)
    for m in OLLAMA_MODELS:
        sh(f"ollama pull {m}")
    sh("ollama list")
print("setup done")

## Day 4 - Retrieve once and freeze it (top 40 per scored question)  
Unzips the Week 1 index from Drive the first time (takes a while).

In [ ]:
!python -m coding_track.steps.w2_freeze_retrieval

## Day 3/4 - Build every prompt once (ImportSnare A.4 template)

In [ ]:
!python -m coding_track.steps.w2_build_prompts

## Day 4 - Pilot: 20 questions, Llama 3.1 8B, clean RAG  
Check: max prompt_eval_count well under 16384, code found, the time estimate. Then export the pilot answers and test them in the test notebook.

In [ ]:
!python -m coding_track.steps.w2_generate_ollama --model llama3.1:8b --setting clean_rag --pilot

In [ ]:
!python -m coding_track.steps.w2_export_for_tests --only llama3.1-8b__clean_rag --out pilot_answers.jsonl
from google.colab import files
files.download('/content/drive/MyDrive/coding_track/week2/pilot_answers.jsonl')

## Days 5-6 - Full clean runs, one model at a time (order: Llama 3.1 8B, Qwen2.5-Coder 7B, Llama 3.2 3B)  
If the session drops: run setup, then the same cell; finished questions are skipped.

In [ ]:
!python -m coding_track.steps.w2_generate_ollama --model llama3.1:8b --setting no_retrieval
!python -m coding_track.steps.w2_generate_ollama --model llama3.1:8b --setting clean_rag

Optional anchor run (compare with CodeRAG-Bench's published no-retrieval numbers on the full DS-1000 and ODEX): 

In [ ]:
!python -m coding_track.steps.w2_generate_ollama --model llama3.1:8b --setting no_retrieval --question-set anchor

In [ ]:
!python -m coding_track.steps.w2_generate_ollama --model qwen2.5-coder:7b --setting no_retrieval
!python -m coding_track.steps.w2_generate_ollama --model qwen2.5-coder:7b --setting clean_rag

In [ ]:
!python -m coding_track.steps.w2_generate_ollama --model llama3.2:3b --setting no_retrieval
!python -m coding_track.steps.w2_generate_ollama --model llama3.2:3b --setting clean_rag

## GPT-5.4 mini through the Batch API  
Add your key in Colab Secrets (key icon) as `OPENAI_API_KEY` first. Probe once, then submit both settings; collect when completed.

In [ ]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
!python -m coding_track.steps.w2_openai_batch probe

In [ ]:
!python -m coding_track.steps.w2_openai_batch submit --setting no_retrieval
!python -m coding_track.steps.w2_openai_batch submit --setting clean_rag

In [ ]:
!python -m coding_track.steps.w2_openai_batch status --setting no_retrieval
!python -m coding_track.steps.w2_openai_batch status --setting clean_rag

In [ ]:
!python -m coding_track.steps.w2_openai_batch collect --setting no_retrieval
!python -m coding_track.steps.w2_openai_batch collect --setting clean_rag

## Day 6 - Export all answers for the test notebook

In [ ]:
!python -m coding_track.steps.w2_export_for_tests
from google.colab import files
files.download('/content/drive/MyDrive/coding_track/week2/answers_for_tests.jsonl')

## Day 7 - After the test notebook: upload its newest `test_outputs.zip` (it is unzipped into Drive `coding_track/week2/test_results/`), then score

In [ ]:
import os, shutil
from google.colab import files
dst = '/content/drive/MyDrive/coding_track/week2/test_results'
os.makedirs(dst, exist_ok=True)
for name in files.upload():
    if name.endswith('.zip'):
        shutil.unpack_archive(name, dst, 'zip')
        os.remove(name)
    else:
        shutil.move(name, os.path.join(dst, name))
print(sorted(os.listdir(dst)))

In [ ]:
!python -m coding_track.steps.w2_score

In [ ]:
!python -m coding_track.steps.w2_handcheck